# Tutorial: Time Series Forecasting with Timestral

This notebook follows the structure of Prashant Banerjee's popular Kaggle notebook
[*Tutorial: Time Series Forecasting with Prophet*](https://www.kaggle.com/code/prashant111/tutorial-time-series-forecasting-with-prophet)
(Apache 2.0), on the same **Air Passengers** dataset, but replaces Prophet with **Timestral**, the time-series
foundation model family released by Mistral AI in October 2026.

Each Prophet step is mapped to its Timestral equivalent. Where Prophet has no equivalent (no fitting step, no
changepoints, no built-in decomposition), the notebook explains why and shows what to do instead. At the end, both
models are compared on held-out data.

## Table of Contents

1. Introduction to Timestral
2. Advantages (and limits) of Timestral
3. Installation of Timestral
4. Python API
5. Basic Setup
6. Time Series Forecasting with Timestral
7. Plotting the forecast components
8. From changepoints to context: how much history the model sees
9. Adjusting the forecast: model size and options
10. Bonus: Timestral vs Prophet on held-out data
11. Conclusion
12. References

## 1. Introduction to Timestral

Prophet is a **statistical model you fit on your series**: an additive model of trend + seasonalities + holidays,
whose parameters are estimated every time you call `fit`.

Timestral is a **pretrained foundation model**: a transformer trained by Mistral on a large corpus of (mostly
synthetic) time series. It is used *zero-shot*: there is no `fit`. You pass the history, and in a single forward
pass it returns a probabilistic forecast (21 quantiles per step). It recognises trend, seasonality and level shifts
from the context it is given, much as a language model continues a text.

* Sizes: 5M, 28M, 123M and 209M parameters (same architecture).
* Probabilistic: quantiles from 1% to 99%.
* Univariate or multivariate, with optional past-only and known-future covariates.
* Up to 32,768 steps of context; CPU is enough.

## 2. Advantages (and limits) of Timestral

**Advantages**

1. **No training, no tuning**: one call, sensible defaults; the same model serves thousands of series.
2. **Fast**: milliseconds per series on CPU, and it batches many series in one call.
3. **Calibrated uncertainty**: a full set of quantiles rather than a single interval.
4. **Strong benchmark accuracy**: #1 among pretrained/zero-shot models on GIFT-Eval (CRPS) and fev-bench
   at release (see the companion "checking the claims" notebook for the nuances).
5. **Handles missing values** (NaN) and series of different lengths.
6. **Covariates**: promotions, prices, calendar or weather forecasts can be passed when known.

**Limits, compared with Prophet**

1. **No interpretable components**: no trend/seasonality decomposition, no changepoints to inspect.
2. **Fewer "domain knowledge" knobs**: you influence it through the data you give it (context, covariates), not
   through priors.
3. **No timestamps**: it reads an ordered sequence of equally spaced values; calendar handling is up to you.
4. **Model weights licence**: Modified MIT (check the model card before commercial use).

## 3. Installation of Timestral

Timestral is installed from GitHub; the weights are downloaded from Hugging Face on first use.

In [ ]:
# Run once (uncomment). Prophet is only needed for the comparison in section 10.
%pip install -q "timestral @ git+https://github.com/mistralai/timestral@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff" \
                pandas matplotlib statsmodels prophet

## 4. Python API

Where Prophet follows the scikit-learn `fit` / `predict` pattern on a `ds` / `y` DataFrame, Timestral has a single
method:

```python
forecaster = TimestralForecaster.from_pretrained("mistralai/Timestral-209M")
fc = forecaster.forecast(values, prediction_length=36)   # values: array of the history, oldest first
fc.median            # point forecast         ~ Prophet's yhat
fc.quantile(0.05)    # lower bound            ~ yhat_lower
fc.quantile(0.95)    # upper bound            ~ yhat_upper
```

Because the model does not see dates, we will keep the `ds` / `y` convention for our DataFrame and write a small
helper that rebuilds a Prophet-style output table (`ds`, `yhat`, `yhat_lower`, `yhat_upper`) with future dates.

## 5. Basic Setup

### Import libraries

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from timestral import TimestralForecaster

%matplotlib inline
plt.style.use("fivethirtyeight")

### Import data

On Kaggle the dataset is attached as `/kaggle/input/air-passengers/AirPassengers.csv`; elsewhere the same data is
downloaded from a public GitHub mirror.

In [ ]:
KAGGLE_FILE = "/kaggle/input/air-passengers/AirPassengers.csv"
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"

df = pd.read_csv(KAGGLE_FILE if os.path.exists(KAGGLE_FILE) else MIRROR)

### Preview dataset

In [ ]:
df.head()

We rename the passenger column (called `#Passengers` on Kaggle and `Passengers` on the mirror) to `AirPassengers`.

In [ ]:
df.rename(columns={"#Passengers": "AirPassengers", "Passengers": "AirPassengers"}, inplace=True)

### Summary of dataset

In [ ]:
df.info()

144 monthly observations, January 1949 to December 1960, no missing values. The `Month` column is text, so we
convert it to datetime, and we keep Prophet's naming convention (`ds` for time, `y` for the value) so the two
notebooks are easy to compare.

In [ ]:
df["Month"] = pd.DatetimeIndex(df["Month"])
df = df.rename(columns={"Month": "ds", "AirPassengers": "y"})
df.dtypes

Timestral assumes **equally spaced** observations. With monthly data this is easy to check: every month must
appear exactly once. (With messier data, reindex to a regular grid and leave missing months as NaN.)

In [ ]:
full_range = pd.date_range(df.ds.min(), df.ds.max(), freq="MS")
assert len(full_range) == len(df) and (df.ds.values == full_range.values).all(), "irregular series"
print(f"{len(df)} regular monthly points from {df.ds.min():%Y-%m} to {df.ds.max():%Y-%m}")

### Visualize the data

In [ ]:
ax = df.set_index("ds").plot(figsize=(12, 8), legend=False)
ax.set_ylabel("Monthly Number of Airline Passengers")
ax.set_xlabel("Date")
plt.show()

Upward trend and a yearly seasonality whose amplitude **grows with the level** (multiplicative seasonality).
Prophet's default additive model struggles with that; let's see how Timestral does without any configuration.

## 6. Time Series Forecasting with Timestral

### Load the model

Prophet starts with `Prophet(interval_width=0.95)`. Timestral starts by loading pretrained weights; the interval
width is chosen *after* forecasting, by picking quantiles.

In [ ]:
MODEL_ID = "mistralai/Timestral-209M"   # or Timestral-5M / -28M / -123M for faster inference
device = "cuda" if torch.cuda.is_available() else "cpu"

my_model = TimestralForecaster.from_pretrained(MODEL_ID, device=device)
print(f"{MODEL_ID} loaded on {device}; quantile levels: {my_model.quantile_levels}")

There is no `fit` step: the 144 values are the model's *context*.

### A Prophet-style `predict`

The helper below plays the role of `make_future_dataframe(periods=36, freq="MS")` + `predict`: it forecasts
`periods` steps, rebuilds the future dates and returns `ds`, `yhat`, `yhat_lower`, `yhat_upper`.

Timestral outputs fixed quantile levels (0.01, 0.05, 0.10, …, 0.95, 0.99), so a 95% interval needs the 2.5% and
97.5% quantiles, which we obtain by linear interpolation between neighbouring levels.

In [ ]:
def quantile_at(fc, level, series=0, variate=0):
    """Forecast quantile at any level in [0.01, 0.99], interpolated between the model's levels."""
    levels = np.asarray(fc.quantile_levels)
    q = fc.quantiles[series, variate].cpu().numpy()          # (horizon, n_levels)
    return np.array([np.interp(level, levels, row) for row in q])


def timestral_predict(model, df, periods, freq, interval_width=0.95, **forecast_kwargs):
    """Prophet-like output for a ds/y DataFrame: one row per future date."""
    history = df.sort_values("ds")["y"].to_numpy(dtype=np.float32)
    fc = model.forecast(history, prediction_length=periods, **forecast_kwargs)
    alpha = (1 - interval_width) / 2
    future_ds = pd.date_range(df["ds"].max(), periods=periods + 1, freq=freq)[1:]
    out = pd.DataFrame({
        "ds": future_ds,
        "yhat": fc.median[0, 0].cpu().numpy(),
        "yhat_lower": quantile_at(fc, alpha),
        "yhat_upper": quantile_at(fc, 1 - alpha),
    })
    return out, fc


forecast, fc = timestral_predict(my_model, df, periods=36, freq="MS", interval_width=0.95)
forecast.head()

* `ds`: the date of the forecasted value
* `yhat`: the point forecast (median of the predictive distribution)
* `yhat_lower` / `yhat_upper`: the bounds of the 95% interval

Two differences with Prophet's output:

1. Prophet's `predict` also returns **in-sample** fitted values for the history; Timestral only forecasts the
   future (there is no fitted model to evaluate in-sample).
2. The output is **deterministic**: the same input always gives the same forecast (Prophet's intervals come from
   random sampling and change slightly between runs).

### Plot the forecast

Equivalent of `my_model.plot(forecast, uncertainty=True)`: observations as black dots, forecast as a line, and
shaded intervals. We show three nested intervals since we have the full set of quantiles.

In [ ]:
def plot_forecast(df, forecast, fc, title="", ax=None):
    ax = ax or plt.subplots(figsize=(12, 7))[1]
    ax.plot(df["ds"], df["y"], "k.", label="observed")
    for lo, hi, a in [(0.025, 0.975, 0.15), (0.1, 0.9, 0.25), (0.25, 0.75, 0.4)]:
        ax.fill_between(forecast["ds"], quantile_at(fc, lo), quantile_at(fc, hi), color="#0072B2", alpha=a,
                        lw=0, label=f"{(hi - lo):.0%} interval")
    ax.plot(forecast["ds"], forecast["yhat"], color="#0072B2", lw=2, label="forecast (median)")
    ax.set_xlabel("ds")
    ax.set_ylabel("y")
    ax.set_title(title, fontsize=13)
    ax.legend(loc="upper left", fontsize=10)
    return ax


plot_forecast(df, forecast, fc, title=f"{MODEL_ID.split('/')[-1]}: 36 months ahead")
plt.show()

Without being told anything about the series, the model continues the trend and reproduces a seasonal pattern
whose amplitude keeps growing, and the intervals widen with the horizon.

## 7. Plotting the forecast components

Prophet's `plot_components` shows the model's own trend and seasonal terms. **Timestral has no such terms**: it is
a neural network, not an additive model, so there is nothing to read off directly.

What we can do is decompose the series *after the fact*: run an STL decomposition (statsmodels) on the history
followed by the forecast median. This is a **diagnostic** of what the forecast looks like (does it continue the
trend? keep the seasonal shape?), not an explanation of how the model computed it. Because the seasonality is
multiplicative, we decompose the logarithm.

In [ ]:
from statsmodels.tsa.seasonal import STL

combined = pd.concat([df[["ds", "y"]], forecast.rename(columns={"yhat": "y"})[["ds", "y"]]]).set_index("ds")["y"]
stl = STL(np.log(combined), period=12, robust=True).fit()
split = df["ds"].max()

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (name, comp) in zip(axes, [("trend (log scale)", stl.trend), ("yearly seasonality (log scale)", stl.seasonal),
                                   ("remainder", stl.resid)]):
    ax.plot(comp[comp.index <= split], color="k", lw=1.5, label="history")
    ax.plot(comp[comp.index >= split], color="#0072B2", lw=2, label="forecast")
    ax.set_title(name, fontsize=12)
axes[0].legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly profile, as in Prophet's "yearly" panel: average seasonal effect per calendar month
season = pd.Series(np.exp(stl.seasonal.values), index=stl.seasonal.index)
profile = pd.DataFrame({
    "history": season[season.index <= split].groupby(season[season.index <= split].index.month).mean(),
    "forecast": season[season.index > split].groupby(season[season.index > split].index.month).mean(),
})
ax = (100 * (profile - 1)).plot(kind="bar", figsize=(12, 5), color=["#3e4c59", "#0072B2"])
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], rotation=0)
ax.set_ylabel("% above / below trend")
ax.set_title("Yearly seasonal profile: history vs forecast", fontsize=13)
plt.show()

As in the Prophet notebook, the peak travel months are **July and August**. If the "forecast" bars match the
"history" bars, the model has carried the seasonal shape forward correctly. Unlike the Prophet notebook, there is no
weekly panel: with monthly data a weekly seasonality cannot be observed (Prophet fitted one anyway, which is an
artefact).

## 8. From changepoints to context: how much history the model sees

Prophet models the trend as a piecewise-linear function with 25 potential **changepoints** in the first 80% of the
data, controlled by `changepoint_range` and `n_changepoints`.

Timestral has no changepoints. Changes in trajectory are inferred from the **context**: the window of history it
is given. The closest equivalent knob is therefore the context length. A short context makes the model focus on
the recent regime (like putting changepoints near the end); a long context lets it use the whole history.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, n_months in zip(axes.flat, [24, 48, 96, len(df)]):
    ctx = df.tail(n_months)
    f, fc_ctx = timestral_predict(my_model, ctx, periods=36, freq="MS")
    plot_forecast(df, f, fc_ctx, title=f"context = last {n_months} months", ax=ax)
    ax.axvspan(ctx["ds"].min(), ctx["ds"].max(), color="#e8590c", alpha=0.06)
    ax.get_legend().remove()
plt.tight_layout()
plt.show()

The orange background marks the history actually seen. With 24 months the model sees only two seasonal cycles and
has less evidence about the trend, so its intervals are wider; with the full history it is more confident.

**Ignoring part of the history.** If an early period is known to be irrelevant (a regime change, a data
migration, the Covid years for travel data), you can hide it without deleting it, using `observation_mask`
(`False` = treat as missing):

In [ ]:
mask = np.ones(len(df), dtype=bool)
mask[(df["ds"] < "1953-01-01").to_numpy()] = False          # pretend the first 4 years are unreliable
f_masked, fc_masked = timestral_predict(my_model, df, periods=36, freq="MS", observation_mask=mask)
print("mean forecast, full history  :", round(forecast["yhat"].mean(), 1))
print("mean forecast, 1949-52 masked:", round(f_masked["yhat"].mean(), 1))

## 9. Adjusting the forecast: model size and options

Prophet lets you make the trend more or less flexible with `changepoint_prior_scale` (0.08 = more flexible,
0.001 = stiffer). Timestral has no prior to tune; the levers are:

| Prophet knob | Timestral lever |
|---|---|
| `changepoint_prior_scale`, `n_changepoints` | model size; context length (section 8) |
| `interval_width` | which quantiles you read (`quantile_at(fc, 0.025)`, …) |
| `seasonality_mode="multiplicative"` | nothing to do: learned from the data |
| `add_regressor`, holidays | `known_covariates`, `categorical_known_covariates` |
| `growth="logistic"` floors / caps | `clip_nonnegative=True` (default) floors at 0 for non-negative series |

### Model size

The four sizes share the same training recipe. Larger models are more accurate on average across benchmarks, at a
higher cost; on a short, clean series like this one, differences are usually small.

In [ ]:
sizes = ["mistralai/Timestral-5M", "mistralai/Timestral-28M", "mistralai/Timestral-123M", "mistralai/Timestral-209M"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, size in zip(axes.flat, sizes):
    m = my_model if size == MODEL_ID else TimestralForecaster.from_pretrained(size, device=device)
    f, fc_size = timestral_predict(m, df, periods=36, freq="MS")
    plot_forecast(df, f, fc_size, title=size.split("/")[-1], ax=ax)
    ax.get_legend().remove()
plt.tight_layout()
plt.show()

### Options of `forecast`

* `symmetric_averaging=True` (default) averages the forecast of the series with the mirrored forecast of its
  negation, making ups and downs symmetric. `False` halves the compute.
* `clip_nonnegative=True` (default) floors forecasts at 0 for series that have never been negative.

In [ ]:
f_fast, _ = timestral_predict(my_model, df, periods=36, freq="MS", symmetric_averaging=False)
print("max |difference| in yhat with symmetric averaging off:",
      round(float(np.abs(f_fast["yhat"] - forecast["yhat"]).max()), 2), "passengers")

## 10. Bonus: Timestral vs Prophet on held-out data

The Prophet tutorial never checks its forecast against reality. Here we hide the last **24 months** (1959-1960),
forecast them from 1949-1958 with both tools, and compare:

* **MAE** and **MAPE** of the point forecast;
* **coverage** of the 95% interval (should be close to 95%);
* **interval width** (narrower is better, *if* coverage holds).

Prophet is run twice: with its defaults (as in the Kaggle notebook) and with `seasonality_mode="multiplicative"`,
which suits this series.

In [ ]:
HOLDOUT = 24
train, test = df.iloc[:-HOLDOUT], df.iloc[-HOLDOUT:]
results, preds = [], {}


def score(name, yhat, lower, upper):
    y = test["y"].to_numpy()
    preds[name] = (yhat, lower, upper)
    results.append({"model": name, "MAE": np.mean(np.abs(y - yhat)), "MAPE %": 100 * np.mean(np.abs(y - yhat) / y),
                    "95% coverage %": 100 * np.mean((y >= lower) & (y <= upper)),
                    "mean interval width": np.mean(upper - lower)})


f_t, _ = timestral_predict(my_model, train, periods=HOLDOUT, freq="MS")
score(MODEL_ID.split("/")[-1], f_t.yhat.values, f_t.yhat_lower.values, f_t.yhat_upper.values)

try:
    import logging
    from prophet import Prophet
    logging.getLogger("cmdstanpy").disabled = True
    for label, kw in [("Prophet (default)", {}), ("Prophet (multiplicative)", {"seasonality_mode": "multiplicative"})]:
        p = Prophet(interval_width=0.95, **kw).fit(train)
        fp = p.predict(p.make_future_dataframe(periods=HOLDOUT, freq="MS")).tail(HOLDOUT)
        score(label, fp.yhat.values, fp.yhat_lower.values, fp.yhat_upper.values)
except ImportError:
    print("Prophet is not installed: `pip install prophet` to run the comparison.")

pd.DataFrame(results).set_index("model").round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(df["ds"], df["y"], "k.-", lw=0.8, label="observed")
colors = {"Prophet (default)": "#7f7f7f", "Prophet (multiplicative)": "#009E73"}
for name, (yhat, lo, hi) in preds.items():
    c = colors.get(name, "#0072B2")
    ax.plot(test["ds"], yhat, color=c, lw=2, label=name)
    ax.fill_between(test["ds"], lo, hi, color=c, alpha=0.12, lw=0)
ax.axvline(test["ds"].min(), color="grey", ls="--", lw=1)
ax.set_xlim(pd.Timestamp("1955-01-01"), df["ds"].max() + pd.offsets.MonthBegin(1))
ax.set_title("Hold-out 1959-1960: forecasts and 95% intervals", fontsize=13)
ax.legend(loc="upper left", fontsize=10)
plt.show()

A single short series is an anecdote, not a benchmark: for a broad comparison on 100 tasks, see the companion
notebook *Timestral vs Prophet: checking Mistral's benchmark claims*.

## 11. Conclusion

* With Timestral, forecasting Air Passengers takes **one call and no configuration**: no `fit`, no seasonality
  mode, no changepoint tuning, deterministic output and a full set of quantiles.
* Prophet remains attractive when you need an **interpretable decomposition** or want to inject domain knowledge
  through priors, changepoints and holiday effects. Timestral's equivalents are data-driven: context length,
  `observation_mask` and covariates.
* In practice, the right question is empirical: backtest both on your own series (section 10 is a template), and
  consider the smaller Timestral sizes when you have many series or tight latency budgets.

## 12. References

* Original notebook: Prashant Banerjee, [Tutorial: Time Series Forecasting with Prophet](https://www.kaggle.com/code/prashant111/tutorial-time-series-forecasting-with-prophet) (Apache 2.0); structure and dataset reused here.
* Mistral AI, [Introducing Timestral](https://mistral.ai/news/timestral/) and [mistralai/timestral](https://github.com/mistralai/timestral).
* Model cards: [Hugging Face collection](https://huggingface.co/collections/mistralai/timestral).
* Prophet: [documentation](https://facebook.github.io/prophet/) and Taylor & Letham, *Forecasting at Scale* (PeerJ Preprints, 2017).
* Cleveland et al., *STL: A Seasonal-Trend Decomposition Procedure Based on Loess* (1990), used in section 7.